# Flight Delay Prediction - Data Cleaning

This notebook covers loading and cleaning the raw flight, weather, and geographic/demographic data.

In [ ]:
# Import libraries
import datetime as dt
import pandas as dt
import pandas as pd
import pandasql as ps
import numpy as np

import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0] / "src"))

from flight_delay_prediction.config import SEED, DATASETS_PATH
from flight_delay_prediction.utils import display_NA_summary

## Flight Data

We began with the raw flight data. We individually scraped BTS flight data by month-year and then combined it into a raw .parquet file. 

The raw scraped CSV files can be found in `../data/external/flights` and the combined parquet file can be found in `../data/intermediate/flights.parquet`. We will load and clean this combined dataset before moving on to the other datasets. 

In [24]:
flights_df = pd.read_parquet(DATASETS_PATH + "/flights.parquet")

In [26]:
# Basic info
display_NA_summary(flights_df)

,Column,Data Type,Number of NA Values,Percent NA
24,LateAircraftDelay,float64,394357,80.75
23,SecurityDelay,float64,394357,80.75
22,NASDelay,float64,394357,80.75
21,WeatherDelay,float64,394357,80.75
20,CarrierDelay,float64,394357,80.75
13,ArrDelay,float64,13018,2.67
18,AirTime,float64,13018,2.67
17,ActualElapsedTime,float64,13018,2.67
10,WheelsOn,float64,12239,2.51
11,TaxiIn,float64,12239,2.51


The main information that stands out from the output of `flights_df.info()` is the very low number of non-null values for the last 5 columns. We will need to look at what exactly is causing the NA values for these columns and whether or not we can reasonably impute the NA values. Otherwise, we see some columns with a relatively small (< 10%) amount of missing values like `DepDelay` and `TaxiOut`.

In order to better understand our data, especially the behavior of delays, we calculated some common statistics to describe the distribution of each column.

In [27]:
flights_df.describe()

,FlightId,DOT_ID_Reporting_Airline,CRSDepTime,DepDelay,TaxiOut,WheelsOff,WheelsOn,TaxiIn,CRSArrTime,ArrDelay,...,Diverted,CRSElapsedTime,ActualElapsedTime,AirTime,Distance,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
count,488392.000000,488392.000000,488392.000000,476427.000000,476198.000000,476198.000000,476153.000000,476153.000000,488392.000000,475374.000000,...,488392.000000,488392.000000,475374.000000,475374.000000,488392.000000,94035.000000,94035.000000,94035.000000,94035.000000,94035.000000
mean,244195.500000,20065.295390,1292.753884,13.075508,17.261234,1308.584238,1415.334703,7.514561,1437.158489,7.518152,...,0.001652,149.178987,143.829379,119.064278,892.800550,28.343393,3.500388,12.907215,0.179508,35.982900
std,140986.770673,335.343967,503.500309,67.863894,8.774088,521.880676,547.021215,5.535872,533.334644,69.242305,...,0.040616,63.608243,63.931164,62.623689,588.519116,97.634385,27.082548,29.851321,2.960894,84.952342
min,0.000000,19393.000000,1.000000,-56.000000,1.000000,1.000000,1.000000,1.000000,1.000000,-88.000000,...,0.000000,-56.000000,35.000000,18.000000,80.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,122097.750000,19805.000000,830.000000,-7.000000,12.000000,844.000000,944.000000,5.000000,951.000000,-16.000000,...,0.000000,102.000000,98.000000,72.000000,453.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,244195.500000,19805.000000,1255.000000,-3.000000,15.000000,1305.000000,1436.000000,6.000000,1448.000000,-7.000000,...,0.000000,129.000000,127.000000,101.000000,690.000000,3.000000,0.000000,1.000000,0.000000,3.000000
75%,366293.250000,20416.000000,1700.000000,7.000000,19.000000,1721.000000,1846.000000,9.000000,1852.000000,8.000000,...,0.000000,170.000000,168.000000,141.000000,1013.000000,22.000000,0.000000,17.000000,0.000000,39.000000
max,488391.000000,20452.000000,2359.000000,3403.000000,179.000000,2400.000000,2400.000000,296.000000,2359.000000,3407.000000,...,1.000000,397.000000,584.000000,555.000000,2522.000000,3403.000000,1332.000000,1217.000000,277.000000,2557.000000


Of course, some of these values are not meaningful (for example, `FlightId` and `DOT_ID_Reporting_Airline` represent discrete categories even though they are numbers). We also found that several columns' data types were not properly preserved when they were written to the parquet format (`CRSDepTime`, `CRSArrTime`, etc.). We handle these in later sections.

We will use `ArrDelay` to create our binary target column that indicates whether a flight arrived meaningfully late, so understanding its distribution should provide some useful information. We see that, **on average, flights are about 8 minutes late** and that there are some significant outliers, **with one flight being recorded as over 50 hours late**. 

We also see that the data indicates early flights using negative numbers. For our problem, we do not need to distinguish between early and on-time flights so we can condense anything below 0 to 0 in `ArrDelay`. Interestingly, `ArrDelay` has a standard deviation of 70 minutes, indicating that delay times vary quite drastically across our sample.

In [28]:
flights_df.head(5)

,FlightId,FlightDate,DOT_ID_Reporting_Airline,Tail_Number,Origin,Dest,CRSDepTime,DepDelay,TaxiOut,WheelsOff,...,Diverted,CRSElapsedTime,ActualElapsedTime,AirTime,Distance,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,0,2020-01-01,20409,N655JB,FLL,PHL,2152,-9.0,29.0,2212.0,...,0.0,158.0,166.0,130.0,992.0,NaN,NaN,NaN,NaN,NaN
1,1,2020-01-02,20409,N591JB,FLL,PHL,2152,0.0,14.0,2206.0,...,0.0,158.0,147.0,128.0,992.0,NaN,NaN,NaN,NaN,NaN
2,2,2020-01-03,20409,N657JB,FLL,PHL,2152,-2.0,13.0,2203.0,...,0.0,158.0,143.0,124.0,992.0,NaN,NaN,NaN,NaN,NaN
3,3,2020-01-04,20409,N709JB,FLL,PHL,2152,23.0,11.0,2226.0,...,0.0,158.0,134.0,119.0,992.0,NaN,NaN,NaN,NaN,NaN
4,4,2020-01-05,20409,N627JB,FLL,PHL,2152,-3.0,16.0,2205.0,...,0.0,158.0,153.0,131.0,992.0,NaN,NaN,NaN,NaN,NaN


By examining the first 5 rows of our data, we can see that some columns represent categorical features of the flight. Specifically, `DOT_ID_Reporting_Airline`, `Origin`,

### Cleaning Flight Data

`FlightDate` needs to be converted to a proper datetime column for feature engineering in a later subsection. We also need to address the `Cancelled` and `Diverted` columns to make sure we are only using observations that represent completed flights (discussed below).

In [29]:
flights_df["FlightDate"] = pd.to_datetime(flights_df["FlightDate"])

The next issue to look is the potential inclusion of cancelled or diverted flights in our data. Since our primary goal is to predict whether or not a flight will be delayed upon arrival to Philadelphia International Airport, we are assuming that our data represents flights **that reached Philadelphia International Airport**. Therefore, including flights that may not have reached their destination or may have been scheduled for a different airport but diverted to PHL could add noise to our data. Below, we'll examine the properties of the rows where a flight was either cancelled or diverted (`Cancelled` = 1 or `Diverted` = 1).

In [30]:
# Isolate rows where Cancelled == 1 or Diverted == 1
flights_df[(flights_df["Cancelled"] == 1) | (flights_df["Diverted"] == 1)]

,FlightId,FlightDate,DOT_ID_Reporting_Airline,Tail_Number,Origin,Dest,CRSDepTime,DepDelay,TaxiOut,WheelsOff,...,Diverted,CRSElapsedTime,ActualElapsedTime,AirTime,Distance,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
192,192,2020-01-11,20397,N547NN,GRR,PHL,1225,NaN,NaN,NaN,...,0.0,115.0,NaN,NaN,573.0,NaN,NaN,NaN,NaN,NaN
199,199,2020-01-18,20397,N589NN,GRR,PHL,1225,NaN,NaN,NaN,...,0.0,115.0,NaN,NaN,573.0,NaN,NaN,NaN,NaN,NaN
295,295,2020-01-20,20397,N535EA,IND,PHL,1303,NaN,NaN,NaN,...,0.0,107.0,NaN,NaN,588.0,NaN,NaN,NaN,NaN,NaN
397,397,2020-01-20,20397,N705PS,PVD,PHL,1745,NaN,NaN,NaN,...,0.0,85.0,NaN,NaN,237.0,NaN,NaN,NaN,NaN,NaN
413,413,2020-01-17,20397,N529EA,STL,PHL,1700,NaN,NaN,NaN,...,0.0,129.0,NaN,NaN,814.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
488260,488260,2025-07-31,20416,N698NK,FLL,PHL,2050,NaN,NaN,NaN,...,0.0,160.0,NaN,NaN,993.0,NaN,NaN,NaN,NaN,NaN
488296,488296,2025-07-14,20416,N917NK,MCO,PHL,2047,NaN,NaN,NaN,...,0.0,154.0,NaN,NaN,862.0,NaN,NaN,NaN,NaN,NaN
488313,488313,2025-07-31,20416,N904NK,MCO,PHL,2047,NaN,NaN,NaN,...,0.0,154.0,NaN,NaN,862.0,NaN,NaN,NaN,NaN,NaN
488338,488338,2025-07-10,20416,N965NK,MCO,PHL,535,NaN,NaN,NaN,...,0.0,149.0,NaN,NaN,862.0,NaN,NaN,NaN,NaN,NaN


There are 13,018 rows that represent flights that were ether cancelled or diverted. That makes up almost 3% of our data. Given that we have 400,000+ observations, losing 3% of our data is not too concerning, so we will filter these rows out to create a dataset of **only flights that were scheduled to go to PHL and arrived at PHL**.

In [31]:
# Filter out cancellations and diverted flights
completed_flights_df = (flights_df[(flights_df["Cancelled"] == 0) & (flights_df["Diverted"] == 0)]
                        .copy()
                        .drop(columns = ["Cancelled", "Diverted"]))
print(f"Dropped\
 {100 - round(completed_flights_df.shape[0] / flights_df.shape[0], 2) * 100}%\
 of observations and 2 columns.")
print(f"New data shape: {completed_flights_df.shape[0]} rows X\
 {completed_flights_df.shape[1]} columns.")

Dropped 3.0% of observations and 2 columns.
New data shape: 475374 rows X 23 columns.


Now we can move on to addressing the NA values in our dataset. Before doing anything, we will recalculate our NA statistics for our filtered dataset.

In [32]:
display_NA_summary(completed_flights_df, only_nonzero = True)

,Column,Data Type,Number of NA Values,Percent NA
18,CarrierDelay,float64,381339,80.22
19,WeatherDelay,float64,381339,80.22
20,NASDelay,float64,381339,80.22
21,SecurityDelay,float64,381339,80.22
22,LateAircraftDelay,float64,381339,80.22


It appears that filtering out cancelled and diverted flights took care of all our NA issues excpet for the majority-NA columns. To handle these columns, we should first understand what each column represents:

| Column | Definition |
|--------|------------|
| SecurityDelay | Security delay, **in minutes** |
| LateAircraftDelay | Late aircraft delay, **in minutes** |
| NASDelay | National Air System Delay, **in minutes** |
| WeatherDelay | Weather delay, **in minutes** |
| CarrierDelay | Carrier delay, **in minutes** |


Given the above definitions, a reasonable first assumption is that when a flight is not delayed at all, these features are encoded as NA. However, we cannot just assume this to be the case so we will conduct some checks to see if that is a reasonable assumption. First, we will check to see if some rows have these columns encoded with a value of 0. If they do, our assumption seems less reasonable.

In [33]:
for col in ["SecurityDelay", "LateAircraftDelay", "NASDelay", "WeatherDelay", "CarrierDelay"]:
  lowest = np.min(completed_flights_df[~(completed_flights_df[col].isna())][col])
  print(f"Minimum non-null value in {col}: {lowest}")

Minimum non-null value in SecurityDelay: 0.0
Minimum non-null value in LateAircraftDelay: 0.0
Minimum non-null value in NASDelay: 0.0
Minimum non-null value in WeatherDelay: 0.0
Minimum non-null value in CarrierDelay: 0.0


Given that we can see zeroes in these columns, our initial thought of NAs being for when there are no delays makes less sense. Still, we will thoroughly check our assumption by also looking for some counterexamples:

1. Cases where at least one but less than all of the delay columns is NA
2. Cases where all the delay indicator columns are NA and the flight was delayed.

If the NAs were a result of on-time flights, we would expect there to be no rows for either of these checks.

In [40]:
# Are there any rows where at least one but less than all 5 of the columns are NA?
mask = (
    (completed_flights_df[["CarrierDelay", "WeatherDelay", "NASDelay", 
                           "SecurityDelay", "LateAircraftDelay"]]
     .isna().sum(axis = 1) < 5)
    &
    (completed_flights_df[["CarrierDelay", "WeatherDelay", "NASDelay", 
                           "SecurityDelay", "LateAircraftDelay"]]
     .isna().sum(axis = 1) > 0)
)

print(
    f"There are {completed_flights_df[mask].shape[0]} instances "
    "where at least one but less than all 5 of the columns are NA."
    )

# Are there rows where the flight was delayed but where the delay type columns are NA?
mask = (completed_flights_df["ArrDelay"] > 0) & (
    completed_flights_df["CarrierDelay"].isna() |
    completed_flights_df["WeatherDelay"].isna() |
    completed_flights_df["NASDelay"].isna() |
    completed_flights_df["SecurityDelay"].isna() |
    completed_flights_df["LateAircraftDelay"].isna()
)

print(f"There are {completed_flights_df[mask].shape[0]} instances\
 where a flight was delayed but at least one of the indicator columns\
 was an NA.")

There are 0 instances where at least one but less than all 5 of the columns are NA.
There are 68962 instances where a flight was delayed but at least one of the indicator columns was an NA.


It appears that when a delay indicator column has an NA value, the other delay indicator columns for that row have an NA as well. However, we do see instances where a flight was delayed but there is no information in any of the delay indicator columns. Given the results of these checks, we cannot be confident that NA values in the delay indicator columns can be reasonably filled with 0. Since these columns are more than 80% NA, we will drop them rather than trying to impute their values.

We are choosing to drop these columns for two reasons:
1. At 80% NA, our imputation would likely not be very well-informed and would probably not be adding very accurate information.
2. Given that our model is trying to **predict** whether a flight will be delayed, these columns cannot be included in any predictive model. Including a column that represents how much a particular issue contributed to a delay inherently encodes information about whether a flight was delayed or not, leading to data leakage.

While it would have been informative to explore how much each factor played into flight delays, dropping these columns makes the most sense from both a data and model integrity standpoint. Now that we have decided on how to approach the delay indicator columns, we can officially drop them from our dataset.

In [41]:
completed_flights_df = completed_flights_df.drop(columns = ["SecurityDelay", "LateAircraftDelay",
                                                            "NASDelay", "WeatherDelay", "CarrierDelay"])
completed_flights_df.head(5)

,FlightId,FlightDate,DOT_ID_Reporting_Airline,Tail_Number,Origin,Dest,CRSDepTime,DepDelay,TaxiOut,WheelsOff,WheelsOn,TaxiIn,CRSArrTime,ArrDelay,CRSElapsedTime,ActualElapsedTime,AirTime,Distance
0,0,2020-01-01,20409,N655JB,FLL,PHL,2152,-9.0,29.0,2212.0,22.0,7.0,30,-1.0,158.0,166.0,130.0,992.0
1,1,2020-01-02,20409,N591JB,FLL,PHL,2152,0.0,14.0,2206.0,14.0,5.0,30,-11.0,158.0,147.0,128.0,992.0
2,2,2020-01-03,20409,N657JB,FLL,PHL,2152,-2.0,13.0,2203.0,7.0,6.0,30,-17.0,158.0,143.0,124.0,992.0
3,3,2020-01-04,20409,N709JB,FLL,PHL,2152,23.0,11.0,2226.0,25.0,4.0,30,-1.0,158.0,134.0,119.0,992.0
4,4,2020-01-05,20409,N627JB,FLL,PHL,2152,-3.0,16.0,2205.0,16.0,6.0,30,-8.0,158.0,153.0,131.0,992.0


### Flight Data Feature Engineering

In this section, we will explore what columns to keep or drop and which columns need to be transformed in some way to be usable for our models. We will also create our binary target column by evaluating `ArrDelay` values against a reasonable threshold.

In [42]:
print(f"Remaing columns after cleaning:")
for i in completed_flights_df.columns:
      print(f"  * {i}")

print(f"\nWhile it could potentially be interesting to include information about the specific plane\n\
being used via `Tail_Number`, it isn't feasible given our current data due to the column having \n\
{completed_flights_df["Tail_Number"].nunique()} unique values.")

Remaing columns after cleaning:
  * FlightId
  * FlightDate
  * DOT_ID_Reporting_Airline
  * Tail_Number
  * Origin
  * Dest
  * CRSDepTime
  * DepDelay
  * TaxiOut
  * WheelsOff
  * WheelsOn
  * TaxiIn
  * CRSArrTime
  * ArrDelay
  * CRSElapsedTime
  * ActualElapsedTime
  * AirTime
  * Distance

While it could potentially be interesting to include information about the specific plane
being used via `Tail_Number`, it isn't feasible given our current data due to the column having 
5405 unique values.


Of our remaining columns, we cannot use `DepDelay`, `TaxiOut`, `WheelsOff`, `WheelsOn`, `TaxiIn` or `ActualElapsedTime` due to data leakage concerns. These variables would not be known by our users at Philadelphia International Airport at the time of prediction (before flight departs from origin airport) so we cannot include them in our predictive model. While it may be interesting to explore their relationships with our target variable, we are choosing to drop them to avoid scope creep and only focus on variables that we can use.

We also cannot use `Tail_Number` due to its high cardinality and `FlightId` because it does not confer any useful information. We will also drop these columns.

We **can** use `FlightDate`, `DOT_ID_Reporting_Airline`, `CRSDepTime` (*Scheduled Departure Time*), `CRSArrTime` (*Scheduled Arrival Time*) but we will need to convert them into a usable formats.

Finally `AirTime`, `Distance`, `CRSElapsedTime` (*Scheduled Elapsed Time*) are already usable and are features that will be known at the time of prediction, meaning they are appropriate to include as-is.

In [54]:
# Drop data leakage, useless, and high cardinality columns
engineered_flights_df = completed_flights_df.copy()
engineered_flights_df = engineered_flights_df.drop(columns = ["DepDelay", "TaxiOut", "WheelsOff",
                                                            "WheelsOn", "TaxiIn", "ActualElapsedTime",
                                                              "Tail_Number", "FlightId"])

In [55]:
# Extract Day, Month, Year from Flight Date
engineered_flights_df["Year"] = engineered_flights_df["FlightDate"].dt.year
engineered_flights_df["Month"] = engineered_flights_df["FlightDate"].dt.month
engineered_flights_df["DayOfWeek"] = engineered_flights_df["FlightDate"].dt.dayofweek

# Create indicator features for weekend flights
engineered_flights_df["IsWeekend"] = engineered_flights_df["DayOfWeek"].isin([5,6]).astype(int)

In [56]:
# Use pandasql to extract Season from Year column with CASE WHEN
query = """
SELECT
  *,
  CASE
    WHEN Month IN (9, 10, 11) THEN 'Fall'
    WHEN Month IN (12, 1, 2) THEN 'Winter'
    WHEN Month IN (3, 4, 5) THEN 'Spring'
    WHEN Month IN (6, 7, 8) THEN 'Summer'
  END as Season
FROM engineered_flights_df
"""

engineered_flights_df = ps.sqldf(query, globals())

# Convert FlightDate back to datetime bc pandasql casted it to object
engineered_flights_df["FlightDate"] = pd.to_datetime(engineered_flights_df["FlightDate"])

In [58]:
# Check unique values for DOT_ID_Reporting_Airline
engineered_flights_df["DOT_ID_Reporting_Airline"].unique()

array([20409, 20397, 19930, 19805, 20436, 20416, 19790, 19393, 20398,
       20378, 20304, 19977, 20452, 20363, 20366])

In [60]:
from flight_delay_prediction.features import extract_time_of_day

engineered_flights_df["CRSArr_TimeOfDay"] = engineered_flights_df["CRSArrTime"].apply(extract_time_of_day)
engineered_flights_df["CRSDep_TimeOfDay"] = engineered_flights_df["CRSDepTime"].apply(extract_time_of_day)

engineered_flights_df[["CRSArrTime", "CRSArr_TimeOfDay", "CRSDepTime", "CRSDep_TimeOfDay"]].sample(5)

,CRSArrTime,CRSArr_TimeOfDay,CRSDepTime,CRSDep_TimeOfDay
152493,1239,Afternoon,1102,Morning
281767,2144,Night,2051,Evening
71213,1119,Morning,737,Morning
43952,1513,Afternoon,1332,Afternoon
444081,1245,Afternoon,1138,Morning


Now, we will create our target column for modeling. We chose a 15 minute threshold to indicate if a flight will be meaningfully delayed. We arrived at this threshold value by researching what delay amounts require airport administrators to reroute passengers, change gates, or take other subjective action. Specifically, [OAG Aviation](https://www.oag.com/airline-on-time-performance-defining-late#fifteenmins) and the [Federal Aviation Commission](https://www.aspm.faa.gov/aspmhelp/index/OPSNET__Delays.html) note that airlines commonly benchmark on-time flights as arriving within 15 minutes

In [63]:
from flight_delay_prediction.config import DELAY_THRESHOLD
engineered_flights_df["IsDelayed"] = np.where(engineered_flights_df["ArrDelay"] > 15, 1, 0)

engineered_flights_df[["ArrDelay", "IsDelayed"]].sample(5)

,ArrDelay,IsDelayed
434990,-27.0,0
31986,-27.0,0
216969,12.0,0
292427,162.0,1
163634,11.0,0


Now that we have properly cleaned our data and engineered the relevant columns, we can do one final drop to remove redundant columns. We are still keeping some columns that will not be used in the modelling section because they will be useful for creating EDA visualizations. Given the above context, we only need to drop `CRSDepTime` and `CRSArrTime` since we have better representations of those columns in the form of `CRSDep_TimeOfDay` and `CRSArr_TimeOfDay`.

In [64]:
# Columns to drop - Either too granular, data leakage, or not useful
cleaned_flights_df = engineered_flights_df.copy()
cleaned_flights_df = cleaned_flights_df.drop(columns = ["CRSDepTime", "CRSArrTime"])

## Destination Weather Data

In this section, we will load the daily weather data in Philadelphia for each day in our flights dataset. This data was collected by running a script that called the OpenMeteo API in batches.

The raw scraped CSV files can be found in `../data/external/weather` and the combined parquet file can be found in `../data/intermediate/destination_weather.parquet`.

In [66]:
destination_weather_df = pd.read_parquet(DATASETS_PATH + "/destination_weather.parquet")

In [69]:
print(f"The destination weather dataset has {destination_weather_df.shape[0]} rows and {destination_weather_df.shape[1]} columns.")
print("-" * 80)
print("Destination weather dataset info:")

display_NA_summary(destination_weather_df)

The destination weather dataset has 2039 rows and 17 columns.
--------------------------------------------------------------------------------
Destination weather dataset info:


,Column,Data Type,Number of NA Values,Percent NA
0,dest_time,object,0,0.0
9,dest_wind_direction_10m_dominant,int64,0,0.0
15,dest_precipitation_hours,float64,0,0.0
14,dest_snowfall_sum,float64,0,0.0
13,dest_rain_sum,float64,0,0.0
12,dest_precipitation_sum,float64,0,0.0
11,dest_et0_fao_evapotranspiration,float64,0,0.0
10,dest_shortwave_radiation_sum,float64,0,0.0
8,dest_wind_gusts_10m_max,float64,0,0.0
1,dest_temperature_2m_mean,float64,0,0.0


The output shows that all columns have complete data with no missing values. The `dest_time` column needs to be converted to datetime format for proper joining with the flight data. The dataset contains 14 continuous weather measurements (temperature, wind, precipitation, etc.) and 2 categorical weather indicators (wind direction and weather code). Since this dataset represents daily weather conditions at Philadelphia, we have one record per day covering the same time period as our flight data.


In [70]:
destination_weather_df.describe()

,dest_temperature_2m_mean,dest_temperature_2m_max,dest_temperature_2m_min,dest_apparent_temperature_mean,dest_apparent_temperature_max,dest_apparent_temperature_min,dest_wind_speed_10m_max,dest_wind_gusts_10m_max,dest_wind_direction_10m_dominant,dest_shortwave_radiation_sum,dest_et0_fao_evapotranspiration,dest_precipitation_sum,dest_rain_sum,dest_snowfall_sum,dest_precipitation_hours,dest_weather_code
count,2039.000000,2039.000000,2039.000000,2039.000000,2039.000000,2039.000000,2039.000000,2039.000000,2039.000000,2039.000000,2039.000000,2039.000000,2039.000000,2039.000000,2039.000000,2039.000000
mean,-4.461157,0.566307,-10.636145,-8.385875,-2.582933,-15.115400,9.698774,40.627268,267.609122,18.710628,2.254129,1.452771,0.662334,0.554885,3.565473,35.896027
std,9.759072,9.533580,10.035904,10.429226,10.265303,10.768051,2.228888,13.748574,111.106464,6.799786,1.363208,3.191997,1.742919,1.816909,5.215240,32.129033
min,-28.500000,-25.000000,-33.400000,-33.200000,-29.500000,-39.000000,3.900000,15.800000,0.000000,4.990000,0.210000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-13.200000,-7.600000,-19.300000,-17.750000,-11.400000,-24.500000,8.200000,30.200000,175.500000,12.505000,0.980000,0.000000,0.000000,0.000000,0.000000,3.000000
50%,-4.100000,0.200000,-10.000000,-8.000000,-2.800000,-14.600000,9.400000,38.200000,334.000000,18.500000,1.940000,0.100000,0.000000,0.000000,1.000000,51.000000
75%,4.700000,9.500000,-1.400000,1.400000,6.900000,-5.200000,10.900000,47.900000,346.000000,24.455000,3.420000,1.400000,0.300000,0.210000,5.000000,71.000000
max,15.000000,21.100000,10.200000,11.900000,18.300000,7.400000,25.200000,95.400000,360.000000,34.110000,5.890000,38.500000,20.100000,25.340000,24.000000,75.000000


The temperature values appear to be in Celsius, with mean temperatures around -4°C and ranging from -28.5°C to 15°C. This range is consistent with Philadelphia's climate, which experiences cold winters and warm summers. Wind speeds range from 3.9 to 25.2 m/s, and precipitation measurements show typical variability with most days having minimal precipitation.


In [71]:
destination_weather_df.head(5)

,dest_time,dest_temperature_2m_mean,dest_temperature_2m_max,dest_temperature_2m_min,dest_apparent_temperature_mean,dest_apparent_temperature_max,dest_apparent_temperature_min,dest_wind_speed_10m_max,dest_wind_gusts_10m_max,dest_wind_direction_10m_dominant,dest_shortwave_radiation_sum,dest_et0_fao_evapotranspiration,dest_precipitation_sum,dest_rain_sum,dest_snowfall_sum,dest_precipitation_hours,dest_weather_code
0,2020-01-01,-16.8,-10.9,-22.6,-21.6,-15.3,-27.7,8.0,28.8,338,10.44,0.69,0.0,0.0,0.00,0.0,3
1,2020-01-02,-16.1,-10.4,-21.5,-20.8,-14.7,-26.4,7.9,26.3,326,9.66,0.64,0.0,0.0,0.00,0.0,3
2,2020-01-03,-15.0,-10.4,-19.7,-19.6,-14.8,-24.5,7.1,35.3,332,8.78,0.65,0.0,0.0,0.00,0.0,3
3,2020-01-04,-17.9,-12.2,-23.6,-22.8,-16.9,-28.9,9.1,40.7,354,7.64,0.54,0.0,0.0,0.00,0.0,3
4,2020-01-05,-18.7,-14.0,-25.4,-23.4,-18.9,-30.5,8.8,25.9,13,10.25,0.54,0.2,0.0,0.14,2.0,71


In [73]:
destination_weather_df.columns

Index(['dest_time', 'dest_temperature_2m_mean', 'dest_temperature_2m_max',
       'dest_temperature_2m_min', 'dest_apparent_temperature_mean',
       'dest_apparent_temperature_max', 'dest_apparent_temperature_min',
       'dest_wind_speed_10m_max', 'dest_wind_gusts_10m_max',
       'dest_wind_direction_10m_dominant', 'dest_shortwave_radiation_sum',
       'dest_et0_fao_evapotranspiration', 'dest_precipitation_sum',
       'dest_rain_sum', 'dest_snowfall_sum', 'dest_precipitation_hours',
       'dest_weather_code'],
      dtype='object')

In [74]:
destination_weather_df["dest_weather_code"].unique()

array([ 3, 71, 73,  2,  1,  0, 75, 51, 53, 55, 61, 63, 65])

Looking at the available features, we can see several features that will not be very informative in predicting whether conditions in Philadelphia will contribute to a delayed flight. Specifically, we will drop the following:

| Column | Reason for dropping |
| ------ | ------------------- |
| dest_apparent_temperature_mean, _max, _min | Represent what the temperature "felt like" on a given day which is mostly a result of other variables that we already have |
| dest_wind_direction_10m_dominant | Represents the direction the wind is blowing. While this may be useful if we could compare it to the direction the plane is flying in, we don't have the positional data to calculate that and so it seems better to drop the column. |
| dest_shortwave_radiation_sum | Represents sunlight exposure which has no real effect on whether a flight will be delayed or not. |
| dest_et0_fao_evapotranspiration | This variable is used for agricultural analysis and doesn't convey any information that would help predict if a flight was delayed. |

In [75]:
destination_weather_df = destination_weather_df.drop(columns = ["dest_apparent_temperature_mean",
                                                                "dest_apparent_temperature_max",
                                                                "dest_apparent_temperature_min",
                                                                "dest_wind_direction_10m_dominant",
                                                                "dest_shortwave_radiation_sum",
                                                                "dest_et0_fao_evapotranspiration"])

### Cleaning Destination Weather Data
As seen above, the destination weather dataset has no missing values across all 17 columns. So, we move on to other structural checks starting with looking for duplicate rows.

In [76]:
print(f"Number of duplicate rows: {destination_weather_df.duplicated().sum()}")

Number of duplicate rows: 0


Next, we need to convert `dest_time` to datetime to ensure proper time-based joins later and check for any invalid values in key weather columns.

In [77]:
destination_weather_df["dest_time"] = pd.to_datetime(destination_weather_df["dest_time"])

In [78]:
# Check for negative values in columns that should only contain non-negative values
negative_checks = ['dest_precipitation_sum', 'dest_rain_sum', 'dest_snowfall_sum',
                   'dest_wind_speed_10m_max', 'dest_wind_gusts_10m_max',
                   'dest_precipitation_hours']

print("Checking for invalid negative values:")
for col in negative_checks:
    neg_count = (destination_weather_df[col] < 0).sum()
    if neg_count > 0:
        print(f"\t{col} has {neg_count} negative values")
    else:
        print(f"\t * {col}: No negative values")

# Check unique weather codes
print(f"\nUnique weather codes: {sorted(destination_weather_df['dest_weather_code'].unique().tolist())}")
print(f"Total unique codes: {destination_weather_df['dest_weather_code'].nunique()}")


Checking for invalid negative values:
	 * dest_precipitation_sum: No negative values
	 * dest_rain_sum: No negative values
	 * dest_snowfall_sum: No negative values
	 * dest_wind_speed_10m_max: No negative values
	 * dest_wind_gusts_10m_max: No negative values
	 * dest_precipitation_hours: No negative values

Unique weather codes: [0, 1, 2, 3, 51, 53, 55, 61, 63, 65, 71, 73, 75]
Total unique codes: 13


The destination weather dataset is already clean with no missing values, no duplicates, and no invalid data points. All weather measurements fall within reasonable ranges for Philadelphia's climate, and the datetime conversion was successful. However, the OpenMeteo API returned a large number of weather features, many of which may be correlated with each other or not as informative for our problem. In the next section, we will examine the given features and make decisions about combining columns into more informative features.


### Destination Weather Data Feature Engineering

In this section we create a range feature to condense two temperature features into one.

Since we already have a feature that represents the average weather in PHL, we can combine the min and max temperature features into one range features.

In [79]:
# Create temperature range feature and drop min and max features
destination_weather_df["dest_temperature_2m_range"] = (
    destination_weather_df["dest_temperature_2m_max"] - destination_weather_df["dest_temperature_2m_min"]
    )

destination_weather_df = destination_weather_df.drop(columns = ["dest_temperature_2m_max",
                                                                "dest_temperature_2m_min"])

In [80]:
destination_weather_df.columns

Index(['dest_time', 'dest_temperature_2m_mean', 'dest_wind_speed_10m_max',
       'dest_wind_gusts_10m_max', 'dest_precipitation_sum', 'dest_rain_sum',
       'dest_snowfall_sum', 'dest_precipitation_hours', 'dest_weather_code',
       'dest_temperature_2m_range'],
      dtype='object')

## Origin Weather Data

After processing the destination weather data, we loaded the origin weather dataset which contains daily weather features for all origin airports from which flights departed to Philadelphia. Unlike the destination weather data which only covers PHL, this dataset includes weather information for multiple airports across the US, providing context about departure conditions that may impact flight delays.

The raw CSV file can be found in `../data/external/` and the usable parquet file (used here) can be found in `../data/intermediate/origin_weather.parquet`.

In [81]:
origin_weather_df = pd.read_parquet(DATASETS_PATH + "/origin_weather.parquet")

In [84]:
print(f"The origin weather dataset has {origin_weather_df.shape[0]} rows and {origin_weather_df.shape[1]} columns.")
print("-" * 80)
print("Origin weather dataset info:")
display_NA_summary(origin_weather_df)

The origin weather dataset has 263031 rows and 18 columns.
--------------------------------------------------------------------------------
Origin weather dataset info:


,Column,Data Type,Number of NA Values,Percent NA
1,origin_code,object,30585,11.63
0,origin_time,object,0,0.00
16,origin_precipitation_hours,float64,0,0.00
15,origin_snowfall_sum,float64,0,0.00
14,origin_rain_sum,float64,0,0.00
13,origin_precipitation_sum,float64,0,0.00
12,origin_et0_fao_evapotranspiration,float64,0,0.00
11,origin_shortwave_radiation_sum,float64,0,0.00
10,origin_wind_direction_10m_dominant,int64,0,0.00
9,origin_wind_gusts_10m_max,float64,0,0.00


The output shows that this dataset is significantly larger than the destination weather data, which makes sense since we're capturing weather from multiple origin airports rather than just PHL. We have 18 columns total, including an `origin_code` column to identify which airport the weather data corresponds to. Most columns have complete data, though we notice that `origin_code` has some missing values (30,585 null values) that we'll need to investigate. The dataset contains 14 continuous weather measurements and 2 categorical weather indicators, mirroring the structure of the destination weather data.


In [85]:
origin_weather_df.describe()

,origin_temperature_2m_mean,origin_temperature_2m_max,origin_temperature_2m_min,origin_apparent_temperature_mean,origin_apparent_temperature_max,origin_apparent_temperature_min,origin_wind_speed_10m_max,origin_wind_gusts_10m_max,origin_wind_direction_10m_dominant,origin_shortwave_radiation_sum,origin_et0_fao_evapotranspiration,origin_precipitation_sum,origin_rain_sum,origin_snowfall_sum,origin_precipitation_hours,origin_weather_code
count,263031.000000,263031.000000,263031.000000,263031.000000,263031.000000,263031.000000,263031.000000,263031.000000,263031.000000,263031.000000,263031.000000,263031.000000,263031.000000,263031.000000,263031.000000,263031.000000
mean,15.758783,20.785393,11.651212,14.869748,20.436685,10.407091,19.277202,38.604734,188.037623,16.507963,3.308832,3.258063,3.119791,0.097033,3.764971,32.612293
std,9.825056,9.922618,9.974696,12.541954,12.648866,12.691663,7.386885,12.671929,97.709691,7.304787,1.812693,7.846629,7.773828,0.761065,5.250050,28.324240
min,-33.200000,-29.700000,-38.200000,-38.100000,-34.500000,-43.200000,2.300000,5.000000,0.000000,0.380000,0.090000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,8.700000,14.000000,4.200000,5.200000,11.100000,0.400000,13.900000,29.900000,106.000000,10.820000,1.800000,0.000000,0.000000,0.000000,0.000000,3.000000
50%,17.600000,22.900000,13.100000,16.500000,22.600000,11.400000,18.000000,36.700000,197.000000,16.830000,3.220000,0.100000,0.100000,0.000000,1.000000,51.000000
75%,23.900000,28.500000,19.900000,25.400000,30.600000,21.200000,23.200000,45.700000,266.000000,22.620000,4.660000,2.600000,2.300000,0.000000,6.000000,61.000000
max,40.700000,49.000000,34.700000,39.900000,48.300000,35.200000,110.100000,183.200000,360.000000,33.130000,15.080000,232.800000,232.800000,42.560000,24.000000,75.000000


The temperature values are in Celsius, with mean temperatures around 16°C and ranging from -33.2°C to 40.7°C. This wider temperature range compared to the destination data makes sense since we're capturing weather from airports across diverse geographic regions - from colder northern states to warmer southern locations. Wind speeds and precipitation patterns show similar variability across the different origin locations.


In [86]:
origin_weather_df.head(5)

,origin_time,origin_code,origin_temperature_2m_mean,origin_temperature_2m_max,origin_temperature_2m_min,origin_apparent_temperature_mean,origin_apparent_temperature_max,origin_apparent_temperature_min,origin_wind_speed_10m_max,origin_wind_gusts_10m_max,origin_wind_direction_10m_dominant,origin_shortwave_radiation_sum,origin_et0_fao_evapotranspiration,origin_precipitation_sum,origin_rain_sum,origin_snowfall_sum,origin_precipitation_hours,origin_weather_code
0,2020-01-01,FLL,18.9,23.7,14.4,19.2,24.5,13.3,11.3,20.5,351,15.43,2.79,0.0,0.0,0.0,0.0,1
1,2020-01-01,FXE,18.9,23.7,14.4,19.2,24.5,13.3,11.3,20.5,351,15.43,2.79,0.0,0.0,0.0,0.0,1
2,2020-01-02,FLL,21.4,25.5,17.2,22.5,27.2,17.4,16.3,31.3,111,14.55,2.88,0.0,0.0,0.0,0.0,3
3,2020-01-02,FXE,21.4,25.5,17.2,22.5,27.2,17.4,16.3,31.3,111,14.55,2.88,0.0,0.0,0.0,0.0,3
4,2020-01-03,FLL,24.8,27.0,22.9,26.6,28.1,25.2,25.4,46.1,160,12.22,2.86,0.0,0.0,0.0,0.0,3


Looking at the first few rows, we can see that the data includes weather records from different origin airports (FLL and FXE shown here), with each airport having daily weather measurements.


Using the same reasoning as in the destination weather data section, we will drop useless and redundant columns before moving forward with the cleaning.

In [87]:
origin_weather_df = origin_weather_df.drop(columns = ["origin_apparent_temperature_mean",
                                                      "origin_apparent_temperature_max",
                                                      "origin_apparent_temperature_min",
                                                      "origin_wind_direction_10m_dominant",
                                                      "origin_shortwave_radiation_sum",
                                                      "origin_et0_fao_evapotranspiration"])

### Cleaning Origin Weather Data

In this section, we will address some red flags we saw above and explore other aspects of the data to make sure that it is usable. We will start with address the 30,585 missing `origin_code`s. We must make sure the `origin_code` column is complete and contains entries for every unique origin airport in the flights data because `origin_code` is the key identifier that tells us which airport the weather data belongs to. Without this information, we cannot properly join this weather data to our flight records. All other weather measurement columns are complete with no missing values.


In [89]:
origin_weather_df[origin_weather_df["origin_code"].isna()]

,origin_time,origin_code,origin_temperature_2m_mean,origin_temperature_2m_max,origin_temperature_2m_min,origin_wind_speed_10m_max,origin_wind_gusts_10m_max,origin_precipitation_sum,origin_rain_sum,origin_snowfall_sum,origin_precipitation_hours,origin_weather_code
6117,2020-01-01,None,8.4,11.4,5.4,21.6,41.4,0.0,0.0,0.0,0.0,0
6118,2020-01-02,None,7.9,14.7,3.0,20.0,37.4,0.0,0.0,0.0,0.0,3
6119,2020-01-03,None,12.6,18.4,8.8,23.7,40.7,4.0,4.0,0.0,11.0,53
6120,2020-01-04,None,15.2,16.9,13.8,19.8,45.4,31.0,31.0,0.0,20.0,63
6121,2020-01-05,None,8.2,15.2,4.8,35.4,64.1,6.0,6.0,0.0,9.0,61
...,...,...,...,...,...,...,...,...,...,...,...,...
216129,2025-07-27,None,32.8,39.3,26.7,12.6,31.3,0.0,0.0,0.0,0.0,1
216130,2025-07-28,None,32.2,38.3,27.0,17.0,43.2,4.9,4.9,0.0,3.0,63
216131,2025-07-29,None,29.7,36.4,24.9,13.0,29.9,0.2,0.2,0.0,1.0,51
216132,2025-07-30,None,27.7,32.6,24.3,10.8,23.4,8.0,8.0,0.0,10.0,63


In [90]:
print(f"Number of duplicate rows: {origin_weather_df.duplicated().sum()}")

Number of duplicate rows: 0


In [91]:
# Investigate the missing origin_code values
print(f"\nInvestigating missing origin_code values:")
print(f"Rows with missing origin_code: {origin_weather_df['origin_code'].isna().sum()}")
print(f"Percentage of data: {origin_weather_df['origin_code'].isna().sum() / len(origin_weather_df) * 100:.2f}%")

# Check if there are any patterns in the missing data
missing_origin_df = origin_weather_df[origin_weather_df['origin_code'].isna()]
print(f"\nDate range of missing origin_code values:")
print(f"First occurrence: {missing_origin_df['origin_time'].min()}")
print(f"Last occurrence: {missing_origin_df['origin_time'].max()}")


Investigating missing origin_code values:
Rows with missing origin_code: 30585
Percentage of data: 11.63%

Date range of missing origin_code values:
First occurrence: 2020-01-01
Last occurrence: 2025-07-31


Since we cannot reliably impute airport codes (there's no way to determine which airport the weather data belongs to without the code), and these missing values represent a significant portion of the data where we cannot establish the origin-flight relationship, we will drop these rows. While this means losing about 12% of our origin weather data, keeping records without airport identifiers would be meaningless for our analysis since we need to match weather to specific flights. 

It does appear that these missing values could just be an artifact of the API script since they range across the entire dataset. Due to time and resource constraints, we elected to move forward with removing the NA values and planned to double back to the API script only if necessary.


In [92]:
# Drop rows with missing origin_code
origin_weather_df = origin_weather_df[origin_weather_df['origin_code'].notna()].copy()
print(f"\nAfter dropping rows with missing origin_code:")
print(f"New shape: {origin_weather_df.shape[0]} rows X {origin_weather_df.shape[1]} columns")
print(f"Dropped {100 - (origin_weather_df.shape[0] / 263031 * 100):.1f}% of observations")



After dropping rows with missing origin_code:
New shape: 232446 rows X 12 columns
Dropped 11.6% of observations


After cleaning the `origin_code` column, we will convert `origin_time` to pd.datetime in order to allow for temporal joins. We will also verify data quality for checking for any invalid weather values.

In [93]:
origin_weather_df["origin_time"] = pd.to_datetime(origin_weather_df["origin_time"])


In [94]:
# Check for negative values in columns that should only contain non-negative values
negative_checks = ['origin_precipitation_sum', 'origin_rain_sum', 'origin_snowfall_sum',
                   'origin_wind_speed_10m_max', 'origin_wind_gusts_10m_max',
                   'origin_precipitation_hours']

print("Checking for invalid negative values:")
for col in negative_checks:
    neg_count = (origin_weather_df[col] < 0).sum()
    if neg_count > 0:
        print(f"  ⚠ {col} has {neg_count} negative values")
    else:
        print(f"  ✓ {col}: No negative values")

# Check unique weather codes
print(f"\nUnique weather codes: {sorted(origin_weather_df['origin_weather_code'].unique())}")
print(f"Total unique codes: {origin_weather_df['origin_weather_code'].nunique()}")

# Check number of unique origin airports
print(f"\nNumber of unique origin airports: {origin_weather_df['origin_code'].nunique()}")
print(f"Origin airports: {sorted(origin_weather_df['origin_code'].unique())}")


Checking for invalid negative values:
  ✓ origin_precipitation_sum: No negative values
  ✓ origin_rain_sum: No negative values
  ✓ origin_snowfall_sum: No negative values
  ✓ origin_wind_speed_10m_max: No negative values
  ✓ origin_wind_gusts_10m_max: No negative values
  ✓ origin_precipitation_hours: No negative values

Unique weather codes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(51), np.int64(53), np.int64(55), np.int64(61), np.int64(63), np.int64(65), np.int64(71), np.int64(73), np.int64(75)]
Total unique codes: 13

Number of unique origin airports: 114
Origin airports: ['ACK', 'AGS', 'ALB', 'ATL', 'AUS', 'AVL', 'BDL', 'BGR', 'BHM', 'BNA', 'BOS', 'BQN', 'BTV', 'BUF', 'BWI', 'BZN', 'CAE', 'CAK', 'CHA', 'CHO', 'CHS', 'CLE', 'CLT', 'CMH', 'CRW', 'CVG', 'DAB', 'DAL', 'DAY', 'DCA', 'DEN', 'DFW', 'DSM', 'DTW', 'ECP', 'EGE', 'EWR', 'EYW', 'FLL', 'FWA', 'FXE', 'GRR', 'GSO', 'GSP', 'HHH', 'HOU', 'HVN', 'HYA', 'IAD', 'IAH', 'ILM', 'IND', 'JAX', 'JFK', 'LAS', 'LAX', 'LEX

After removing rows with missing airport codes, the origin weather dataset is now clean and ready for analysis. All weather measurements fall within reasonable ranges for various US locations, and we have complete weather information for 114 different origin airports. The wider temperature range (-33.2°C to 40.7°C) compared to the destination data reflects the geographic diversity of origin airports across the country.


In [95]:
# Final verification
print("\nData Quality Summary:")
print(f"✓ Removed {30585} rows with missing origin_code")
print(f"✓ No remaining missing values in weather measurements")
print(f"✓ No duplicates")
print(f"✓ All data types correct")
print(f"✓ Temperature ranges reasonable for diverse US locations (-33.2°C to 40.7°C)")
print(f"✓ No invalid negative values in precipitation/wind data")
print(f"✓ Wind direction within valid range (0-360°)")
print(f"✓ Dataset covers {origin_weather_df['origin_code'].nunique()} unique origin airports")
print(f"✓ Dataset ready for joining with flight data")


Data Quality Summary:
✓ Removed 30585 rows with missing origin_code
✓ No remaining missing values in weather measurements
✓ No duplicates
✓ All data types correct
✓ Temperature ranges reasonable for diverse US locations (-33.2°C to 40.7°C)
✓ No invalid negative values in precipitation/wind data
✓ Wind direction within valid range (0-360°)
✓ Dataset covers 114 unique origin airports
✓ Dataset ready for joining with flight data


### Origin Weather Data Feature Engineering

Since we have th same weather variables as `destination_weather_df`, we will do the same operations we did in that section (create temperature range column and drop min and max columns).

In [96]:
# Create temperature range feature and drop min and max features
origin_weather_df["origin_temperature_2m_range"] = (
    origin_weather_df["origin_temperature_2m_max"] - origin_weather_df["origin_temperature_2m_min"]
    )

origin_weather_df = origin_weather_df.drop(columns = ["origin_temperature_2m_max",
                                                                "origin_temperature_2m_min"])

## Joining Datasets

In this section, we will join the datasets together and add some supplementary information to create a final dataset that can be used for EDA and modeling.

First, we will join `cleaned_flights_df` with `destination_weather_df` on their respective date columns. Then, we will join that intermediate DataFrame with `origin_weather_df` on both date and origin airport code. In order to preserve every recorded flight, we will use a left join for both joins. If a particular flight does not have weather data for a particular date, the weather features will be missing.

In [97]:
# First join flights with destination_weather
intermediate_df = pd.merge(left = cleaned_flights_df, right = destination_weather_df,
                           how = "left", left_on = "FlightDate", right_on = "dest_time")

flights_weather_df = pd.merge(left = intermediate_df, right = origin_weather_df,
                              how = "left", left_on = ["FlightDate", "Origin"],
                              right_on = ["origin_time", "origin_code"])

In [98]:
# Use function from 2.1.3 to check for NAs in joined df
display_NA_summary(flights_weather_df, only_nonzero=True)

,Column,Data Type,Number of NA Values,Percent NA


It looks like the joined worked perfectly. There are no rows in `cleaned_flights_df` that did not have a corresponding entry in `destination_weather_df` or `origin_weather_df`.

The last thing we will do before beginning EDA is load in some supplementary data that will make some of our categorical columns more readable and allow for more robust visualization. Specifially, we will load in our `us_airports`, `us_city_codes`, and `airline_names` tables to assign latitude, longitudes, and city names to our origin airports and to translate `DOT_ID_Reporting_Airline` values into actual airline names.

In [ ]:
# Load us_airports from GitHub
airports = pd.read_parquet(DATASETS_PATH / "us_airports.parquet")

# Load us_city_codes from GitHub
# cities = fetch_github_data(file_name = "us_city_codes.parquet")

# Load airline_names from GitHub
airlines = pd.read_parquet(DATASETS_PATH / "airline_names.parquet")

# Combine us_airports and us_city_codes into one df
# airports_city = pd.merge(left = airports, right = cities, how = "left",
#                          left_on = "city_code", right_on = "city_code")

Below is a brief look at what our two tables look like. We will select only the columns we need from each and join them onto `flights_weather_df`.

In [104]:
airports.sample(5)

,code,name,latitude,longitude,elevation,city_code
1020,JAS,County,30.93,-94.02,255,JAS
2267,WRB,Robins AFB,32.70,-83.65,337,MCN
1702,PLY,Plymouth,41.37,-86.30,770,PLY
950,HVS,Municipal,34.38,-80.07,206,HVS
771,GEG,Spokane International Airport,47.62,-117.54,2342,GEG


In [105]:
airlines.head(5)

,DOT_Id,Description
0,19393,Southwest Airlines Co.
1,19790,Delta Air Lines Inc.
2,19805,American Airlines Inc.
3,19930,Alaska Airlines Inc.
4,19977,United Air Lines Inc.


For `airports` we only want: `code`, `latitude`, and `longitude`. We will join it with `flights_weather_df` using their respective origin code columns. For `airlines`, we will join it to `flights_weather_df` using the `DOT_ID_Reporting_Airline` column.


Like the last set of joins, we will use left joins here to ensure no flight data loss.

In [106]:
# Join airports_city and flights_weather_df
intermediate_df2 = pd.merge(left = flights_weather_df, right = airports[["code", "latitude",
                                                                              "longitude"]],
                            how = "left", left_on = "Origin", right_on = "code")

# Join airlines and intermediate2
final_cleaned_df = pd.merge(left = intermediate_df2, right = airlines,
                            how = "left", left_on = "DOT_ID_Reporting_Airline", right_on = "DOT_Id")

In [112]:
# Check for NAs
display_NA_summary(final_cleaned_df, only_nonzero=True)

,Column,Data Type,Number of NA Values,Percent NA
37,code,object,9602,2.02
38,latitude,float64,9602,2.02
39,longitude,float64,9602,2.02


It looks like `airports` didn't have a few of the airports that were present in the flight data. This isn't a massive issue since these last two joins are purely for visualization purposes. Below we list the airports that have missing latitudes and longitudes.

In [113]:
final_cleaned_df[final_cleaned_df["code"].isna()]["Origin"].unique()

array(['SJU', 'STT', 'BQN'], dtype=object)

Since there are only 3 missing airports, we can easily manually add in their latitudes and longitudes using Google.

In [114]:
# Manually add missing latitudes and longitudes
final_cleaned_df.loc[final_cleaned_df["Origin"] == "SJU", "latitude"] = 18.44
final_cleaned_df.loc[final_cleaned_df["Origin"] == "SJU", "longitude"] = 66.00

final_cleaned_df.loc[final_cleaned_df["Origin"] == "STT", "latitude"] = 18.34
final_cleaned_df.loc[final_cleaned_df["Origin"] == "STT", "longitude"] = 64.97

final_cleaned_df.loc[final_cleaned_df["Origin"] == "BQN", "latitude"] = 18.50
final_cleaned_df.loc[final_cleaned_df["Origin"] == "BQN", "longitude"] = 67.14

# Check NAs again
display_NA_summary(final_cleaned_df, only_nonzero=True) # NAs are gone (we will drop code)


,Column,Data Type,Number of NA Values,Percent NA
37,code,object,9602,2.02


Finally, we'll trim down our final cleaned DataFrame by removing repeat columns. We will also rename some columns for clarity.

In [115]:
# Drop unnecessary columns and rename columns for clarity
final_cleaned_df = final_cleaned_df.drop(columns = ["DOT_ID_Reporting_Airline", "dest_time",
                                                    "origin_time", "origin_code",
                                                    "code", "DOT_Id"])

final_cleaned_df.rename(
    columns = {
        "latitude":"origin_latitude",
        "longitude":"origin_longitude",
        "Description":"airline_name"
    },
    inplace = True
)

final_cleaned_df.to_parquet(DATASETS_PATH + "/../processed/cleaned_eda.parquet")

## Pre-Modelling Preprocessing & Feature Engineering

In this section, we will cyclically encode `Month` and `DayOfWeek` so that we can accurately represent the 12 months and 7 days of the week. By using sine and cosine, we can numerically assert that December is close to January (despite being 12 and 1) and Sunday is close to Monday (despite being 6 and 0).

In [116]:
model_df = final_cleaned_df.copy()

In [119]:
from flight_delay_prediction.features import cycle_transform

# Transform Month and DayOfWeek columns
model_df["SinMonth"] = model_df["Month"].apply(lambda x: cycle_transform(value = x, day = False, sin = True))
model_df["CosMonth"] = model_df["Month"].apply(lambda x: cycle_transform(value = x, day = False, sin = False))

model_df["SinDay"] = model_df["DayOfWeek"].apply(lambda x: cycle_transform(value = x, day = True, sin = True))
model_df["CosDay"] = model_df["DayOfWeek"].apply(lambda x: cycle_transform(value = x, day = True, sin = False))

Now, we will drop redundant or unusable columns before defining the list of columns we will need to construct our ML pipelines in part 5. By defining which columns need which type of encoding here, we are able to easily repeat code to test different ML models with mostly the same pipeline.

In [120]:
# Drop redundant/unusable columns before encoding
cols_to_drop = ["FlightDate", "Dest", "ArrDelay", "Month", "DayOfWeek", "origin_latitude", "origin_longitude"]
model_df = model_df.drop(columns = cols_to_drop)

# Create list of columns that require One Hot Encoding
oh_encoding_cols = ["Year", "Season", "CRSArr_TimeOfDay", "CRSDep_TimeOfDay"]

# Create list of columns that require Target Encoding
target_encoding_cols = ["Origin", "dest_weather_code", "origin_weather_code", "airline_name"]

# Define target variable and numeric columns that do not need encoding
target = "IsDelayed"

numeric_cols = ["CRSElapsedTime", "AirTime", "Distance", "dest_temperature_2m_mean", "dest_wind_speed_10m_max",
                "dest_wind_gusts_10m_max", "dest_precipitation_sum", "dest_rain_sum", "dest_snowfall_sum",
                "dest_precipitation_hours", "dest_temperature_2m_range", "origin_temperature_2m_mean",
                "origin_wind_speed_10m_max", "origin_wind_gusts_10m_max", "origin_precipitation_sum",
                "origin_rain_sum", "origin_snowfall_sum", "origin_precipitation_hours", "origin_temperature_2m_range",
                "SinMonth", "CosMonth", "SinDay", "CosDay"]

In order to properly evaluate our models, we will split out data into a training and testing set. We will use our training set to fit our models and then evaluate how well they generalize by predicting on the test set. Because we have such a high volume of data, we allocated 30% of our data to the testing set and decided to train our model on the remaining 70%.


Also, since we have removed all autoregressive features from the data, we are treating this as a classical machine learning problem and are randomly splitting our data rather than preserving its original temporal order.

Finally, we also made sure to stratify our samples to ensure that the class ratios remained consistent across our training and testing samples.

In [121]:
model_df.to_parquet(DATASETS_PATH + "/../processed/cleaned_pre_encoding_data.parquet")